# Lekcija 17 - Izrada lokalnih AI agenata s Foundry Local i Qwen

U ovom bilježniku gradite **lokalnog inženjerskog asistenta** koji se u potpunosti izvodi na vašem radnom računalu. Nigdje se ne koristi izvođenje u oblaku. Asistent može:

1. **Pozivati alate** putem Qwen poziva funkcija kroz Foundry Local.
2. **Navesti i čitati datoteke** unutar sandboxirane projektne mape.
3. **Analizirati kod** jednostavnim metrikama.
4. **Pretraživati dokumentaciju** lokalnim RAG-om (Chroma).
5. **Koristiti lokalni MCP server** (nježno preskočeno ako nije konfiguriran).

Kod agenta izgleda gotovo identično kao i u lekcijama s oblaka — samo se klijent endpoint pomiče iz oblaka na `localhost`.


## Postavljanje

Prije pokretanja ovog bilježnika:

1. **Instalirajte Microsoft Foundry Local** (pogledajte [dokumentaciju](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) za vaš OS).
2. **Preuzmite i pokrenite Qwen model:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Instalirajte donje Python pakete.

Sve se izvodi lokalno. Računalo s ~8 GB RAM-a je realna minimum.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Povežite se na Foundry Local

`FoundryLocalManager` preuzima model ako je potrebno, pokreće lokalnu uslugu i daje nam **OpenAI-kompatibilnu točku pristupa**. Zatim standardni OpenAI SDK usmjeravamo na nju. API ključ je lokalni privremeni podatak — bez ikakvih podataka o vjerodajnicama za oblak.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Lokalni alati (pješčanik za operacije s datotekama)

Stvaramo mali uzorak projekta na disku, zatim definiramo alate ograničene na korijen tog projekta. Provjera pješčanika je važna čak i lokalno: alat koji čita proizvoljne putove radi s dozvolama vašeg korisnika i može pristupiti svemu što i vi.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Lokalni RAG s Chromom

Ugrađujemo mali skup isječaka dokumentacije u lokalnu Chroma kolekciju. Chroma radi u procesu i sprema vektore na disk — bez servera, bez oblaka. Alat `search_docs` dohvaća najrelevantnije isječke za upit.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Petlja za pozivanje alata

Sada registriramo alate s modelom koristeći OpenAI šemu alata i pokrećemo standardnu petlju za pozivanje alata — model traži alat, mi ga izvršavamo lokalno, vraćamo rezultat i ponavljamo dok model ne proizvede konačan odgovor. Pouzdano pozivanje funkcija Qwena je ono što omogućuje da ovo radi na uređaju.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. Lokalni MCP (opcionalno)

MCP je transport, a ne cloud usluga — MCP server može raditi kao lokalni proces preko `stdio`. Ćelija ispod prikazuje kako biste se povezali s lokalnim MCP serverom ako imate jedan konfiguriran (na primjer server datotečnog sustava). Preskače se bez problema kada `LOCAL_MCP_COMMAND` nije postavljen, tako da bilježnica i dalje radi od početka do kraja bez njega.

Napomena o sigurnosti: lokalni MCP server radi s dopuštenjima vašeg korisnika. Ograničite ga na direktorij projekta i provjerite njegove izlaze prije nego što na njima djelujete.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Sažetak

Izgradili ste inženjerskog asistenta koji se u potpunosti izvodi na vašem računalu:

- **Foundry Local** poslužuje **Qwen** model iza OpenAI-kompatibilne krajnje točke — tako da kôd agenta odgovara lekcijama iz oblaka.
- **Sandboxed alati** dali su agentu pristup datotekama i analizu koda bez izlaska iz direktorija projekta.
- **Chroma** je pružila **lokalni RAG** nad dokumentacijom.
- **Local MCP** pokazao je kako ponovno koristiti MCP ekosustav izvan mreže.

Niti jednom se nije koristilo izvedenje u oblaku.

### Izazov

Proširite lokalnog agenta da:

1. **Radi s više MCP poslužitelja** — povežite poslužitelja datotečnog sustava i git poslužitelja te dopustite agentu da bira između njih.
2. **Koristi lokalnu memoriju** — trajno pohranite kratku povijest razgovora na disk kako bi asistent pamtio ranije dijelove preko ponovnih pokretanja bilježnice.
3. **Podržava lokalnu višestruku orkestraciju agenata** — dodajte drugog lokalnog agenta (npr. recenzenta) i neka ta dva surađuju na zadatku.

U sljedećoj lekciji naučit ćete kako osigurati implementirane AI agente.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
